In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [6]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def write_story(city:str)->str:
    """write a story about the city"""
    return f"it's always sunny in {city}"

agent = create_agent(
    model = "openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [write_story],
)

stream = agent.stream_events({
    "messages":[{"role":"user","content":"what is the weather in SF"}]
},version="v3")

final_text = ""
for message in stream.messages:
    for delta in message.text:
        print(delta,end="",flush=True)
        final_text += delta
print()
# print(final_text)
    

I don't have access to weather data — I'm not connected to a weather service, so I can't look up current conditions in San Francisco.

However, I do have a tool that can write a **story about a city**. Would you like me to write a story about San Francisco instead? I can capture the fog, the Golden Gate Bridge, the hills, and the vibe of the city in a narrative.

Let me know if you'd like that!


In [10]:
agent_input = {"messages":[{"role":"user","content":"what is the weather in SF"}]}
stream = agent.stream_events(agent_input,version="v3")

for message in stream.messages:
    print(f"{message.node}", end="")
    for delta in message.text:
        print(delta,end="",flush=True)

modelI don't have access to a weather tool, so I'm unable to check the current weather in San Francisco for you. My available tools are limited to writing stories.

However, I can tell you that San Francisco's weather is famously unpredictable—cool, foggy summers (the famous "Karl the Fog"), mild winters, and temperatures generally ranging from the mid-50s to low 70s°F year-round. If you'd like, I could also **write a story about San Francisco** for you. Would you like that?

In [14]:
from langchain.tools import tool
@tool
def get_weather(city:str)->str:
    """get the weather of the city"""
    return f"it's sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
)

stream = agent.stream_events(agent_input,version="v3")
for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")
    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized: {finalized}")
        
stream = agent.stream_events(agent_input,version="v3")
for call in stream.tool_calls:
    print(f"call.tool_name: {call.input}")
    for delta in call.output_deltas:
        print(delta,end="",flush=True)
    print(call.output,call.error)


tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_4b68515826254cb19d25f670', 'name': 'get_weather', 'args': '', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_4b68515826254cb19d25f670', 'name': 'get_weather', 'args': '{', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': 'call_4b68515826254cb19d25f670', 'name': 'get_weather', 'args': '{"city": "San Francisco"}', 'index': 0}
finalized: [{'type': 'tool_call', 'id': 'call_4b68515826254cb19d25f670', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}]
call.tool_name: {'city': 'San Francisco'}
content="it's sunny in San Francisco" name='get_weather' id='d8c49280-d116-441f-8270-3e4150ad8ce6' tool_call_id='call_7cea99f39ec64e68b9a76864' None


In [16]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

@tool
def get_weather(city:str)->str:
    """get the weather of the city"""
    return f"it's sunny in {city}"

weather_agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    name="weather_agent",
)

@tool
def call_weather(query:str)->str:
    """Query the weather agent"""
    result = weather_agent.invoke({"messages":[{"role":"user","content":query}]})
    return result["messages"][-1].text

supervisor = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[call_weather],
    name="supervisor",
)

stream = supervisor.stream_events(
    {"messages":[{"role":"user","content":"what is the weather in SF"}]},
    version="v3",
)

for subagent in stream.subagents:
    print(f"subagent: {subagent.name}",end="")
    for message in subagent.messages:
        for token in message.text:
            print(token,end="",flush=True)
    print()




subagent: weather_agentThe weather in San Francisco is sunny! ☀️


In [18]:
stream = agent.stream_events(agent_input, version="v3")

for snapshot in stream.values:
    print(snapshot)

final_state = stream.output

{'messages': [HumanMessage(content='what is the weather in SF', additional_kwargs={}, response_metadata={}, id='65078889-a998-4f81-ae4b-4a7cee34e221')]}
{'messages': [HumanMessage(content='what is the weather in SF', additional_kwargs={}, response_metadata={}, id='65078889-a998-4f81-ae4b-4a7cee34e221'), AIMessage(content=[{'type': 'reasoning', 'reasoning': 'The user asks about the weather in SF. I\'ll call get_weather with city "SF" or "San Francisco". Let me use "San Francisco".', 'index': 0}, {'type': 'tool_call', 'id': 'chatcmpl-tool-8e8795ce33b08976', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}], additional_kwargs={'reasoning_content': 'The user asks about the weather in SF. I\'ll call get_weather with city "SF" or "San Francisco". Let me use "San Francisco".', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknownunknownunknownunknownunknownunknownunknownunknownunknown', 'index': 0, 'text': 'The user asks about the weather in SF. I\'ll call get_weather w

In [23]:
import asyncio
stream = await agent.astream_events(agent_input, version="v3")

async def consume_messages():
    async for message in stream.messages:
        print(await message.text)

async def consume_tool_calls():
    async for call in stream.tool_calls:
        print(call.tool_name, call.input)

await asyncio.gather(consume_messages(), consume_tool_calls())


get_weather {'city': 'San Francisco'}
It's sunny in San Francisco! ☀️


[None, None]

In [25]:
stream = agent.stream_events(agent_input,version="v3")
for name,item in stream.interleave("messages","tool_calls","values"):
    if name == "messages":
        print(item.text)
    elif name == "tool_calls":
        print(item.tool_name,item.input)
    else:
        print(item)

for event in stream:
    print(event["method"], event["params"]["namespace"], event["params"]["data"])

{'messages': [HumanMessage(content='what is the weather in SF', additional_kwargs={}, response_metadata={}, id='39222d9c-e8fc-4e2f-85ff-5654349a721b')]}

{'messages': [HumanMessage(content='what is the weather in SF', additional_kwargs={}, response_metadata={}, id='39222d9c-e8fc-4e2f-85ff-5654349a721b'), AIMessage(content=[{'type': 'reasoning', 'reasoning': 'The user is asking about the weather in SF. I should get the weather for San Francisco. The city parameter "SF" might not be recognized, but let me use "SF" or "San Francisco". Let me use "San Francisco".', 'index': 0}, {'type': 'tool_call', 'id': 'call_24c5efd31e4d4de8a22d65ca', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}], additional_kwargs={'reasoning_content': 'The user is asking about the weather in SF. I should get the weather for San Francisco. The city parameter "SF" might not be recognized, but let me use "SF" or "San Francisco". Let me use "San Francisco".', 'reasoning_details': [{'type': 'reasoning.text', '

In [ ]:
from langgraph.stream._types import StreamTransformer, ProtocolEvent
from langgraph.stream.stream_channel import StreamChannel

class ToolActivityTransformer(StreamTransformer):
    name = "tool_activity"
    supports_sync = True
    
    def __init__(self, scope: tuple = ()):
        super().__init__(scope)
        self._log = StreamChannel()
    
    def init(self) -> dict:
        return {"tool_activity": self._log}
    
    def process(self, event: ProtocolEvent) -> bool:
        if event["method"] == "messages":
            data = event["params"]["data"]
            self._log.push({"action": "message", "data": data})
        return True
        
stream = agent.stream_events(
    agent_input,
    version="v3",
    transformers=[ToolActivityTransformer],
)

for activity in stream.extensions["tool_activity"]:
    print(activity)

{'action': 'message', 'data': ({'event': 'message-start', 'role': 'ai', 'id': 'lc_run--01a03c02-50cf-73a1-b0e3-5b1285e36679', 'metadata': {'provider': 'openrouter'}}, {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:31f0b29b-7e0b-1c69-9636-af0bfbea7336', 'checkpoint_ns': 'model:31f0b29b-7e0b-1c69-9636-af0bfbea7336', 'ls_provider': 'openrouter', 'ls_model_name': 'deepseek/deepseek-v4-flash-0731', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-openrouter': '0.2.7'}, 'run_id': '01a03c02-50cf-73a1-b0e3-5b1285e36679'})}
{'action': 'message', 'data': ({'event': 'content-block-start', 'index': 0, 'content': {'type': 'reasoning', 'reasoning': ''}}, {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_trig

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class ToolActivityMiddleware(AgentMiddleware):
    transformers = (ToolActivityTransformer,)


agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    middleware=[ToolActivityMiddleware()],
)

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
    ],
)